# StarDist Pretrained Inference (HPC)

This notebook follows the official StarDist pretrained flow from the repo:
- `StarDist2D.from_pretrained(model_name)`
- percentile normalization with `normalize(...)`
- `model.predict_instances(...)`

Recommended model for H&E nuclei: `2D_versatile_he`


In [76]:
import socket
import tensorflow as tf

print('host:', socket.gethostname())
print('tensorflow:', tf.__version__)
print('physical_gpus:', tf.config.list_physical_devices('GPU'))


host: hpctpa3pc0004
tensorflow: 2.15.1
physical_gpus: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [77]:
from pathlib import Path
import numpy as np
import imageio.v3 as iio
from tifffile import imwrite
from csbdeep.utils import normalize
from stardist.models import StarDist2D
from tqdm.auto import tqdm
import cv2

In [78]:
# Update these paths for your dataset
INPUT_DIR = Path('/share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC5/preprocessing/output/RCC_5.svs')
OUTPUT_DIR = Path('/share/lab_teng/trainee/tusharsingh/cell-seg/inference/stardist/outputs/RCC_5')
VIZ_DIR = OUTPUT_DIR / 'visualizations'
FAILED_CSV = OUTPUT_DIR / 'failed.csv'

# Pretrained options include: 2D_versatile_he, 2D_versatile_fluo, 2D_paper_dsb2018, 2D_demo
MODEL_NAME = '2D_versatile_he'

# Normalization percentiles (same style as StarDist scripts/examples)
PNORM = (1, 99.8)

# Optional tiling for larger images, e.g. (2, 2), (4, 4)
N_TILES = None

# Optional thresholds; None = model defaults
PROB_THRESH = None
NMS_THRESH = None

# Save side-by-side visualization image for each input
SAVE_VIZ = True

# Fast visualization options
VIZ_MAX_SIDE = None         # downscale before saving (None to disable)
PNG_COMPRESSION = 1        # 0=largest file/fastest, 9=smallest file/slowest

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

In [79]:
# List available pretrained models
StarDist2D.from_pretrained()

There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None


In [80]:
model = StarDist2D.from_pretrained(MODEL_NAME)
print('loaded model:', MODEL_NAME)

Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
loaded model: 2D_versatile_he


In [81]:
def _ensure_yx_or_yxc(img: np.ndarray) -> np.ndarray:
    """Return image in StarDist-friendly 2D (Y,X) or 3D (Y,X,C) format."""
    img = np.asarray(img)
    if img.ndim == 2:
        return img
    if img.ndim == 3:
        if img.shape[-1] in (1, 3):
            return img
        if img.shape[-1] == 4:
            return img[..., :3]
        if img.shape[0] in (1, 3):
            return np.moveaxis(img, 0, -1)
        if img.shape[0] == 4:
            return np.moveaxis(img[:3], 0, -1)
    raise ValueError(f'Unsupported image shape {img.shape}. Expected (Y,X) or (Y,X,C).')


def predict_one_image(img: np.ndarray):
    """Run StarDist inference on one image array."""
    img = _ensure_yx_or_yxc(img)
    axis_norm = (0, 1) if img.ndim == 3 else None
    img_norm = normalize(img, *PNORM, axis=axis_norm)
    labels, details = model.predict_instances(
        img_norm,
        n_tiles=N_TILES,
        prob_thresh=PROB_THRESH,
        nms_thresh=NMS_THRESH,
    )
    return labels, details


def _to_uint8_rgb(img: np.ndarray) -> np.ndarray:
    """Convert grayscale/RGB float/int image to uint8 RGB."""
    arr = np.asarray(img)
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    elif arr.ndim == 3 and arr.shape[-1] == 1:
        arr = np.repeat(arr, 3, axis=-1)
    elif arr.ndim == 3 and arr.shape[-1] >= 3:
        arr = arr[..., :3]
    else:
        raise ValueError(f'Unsupported image shape for visualization: {arr.shape}')

    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        if arr.max() <= 1.0:
            arr *= 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr


def _compute_outlines_fast(masks: np.ndarray) -> np.ndarray:
    """Fast boundary extraction from instance labels using neighbor differences."""
    m = masks.astype(np.int32, copy=False)
    outlines = np.zeros(m.shape, dtype=bool)

    # Vertical neighbors
    dv = m[1:, :] != m[:-1, :]
    fg_v = (m[1:, :] > 0) | (m[:-1, :] > 0)
    edge_v = dv & fg_v
    outlines[1:, :] |= edge_v
    outlines[:-1, :] |= edge_v

    # Horizontal neighbors
    dh = m[:, 1:] != m[:, :-1]
    fg_h = (m[:, 1:] > 0) | (m[:, :-1] > 0)
    edge_h = dh & fg_h
    outlines[:, 1:] |= edge_h
    outlines[:, :-1] |= edge_h

    return outlines


def mask_overlay_fast(img_rgb: np.ndarray, masks: np.ndarray) -> np.ndarray:
    """Fast colored overlay using LUT mapping (no per-instance Python loops)."""
    h, w = masks.shape
    gray = img_rgb.astype(np.float32).mean(axis=-1)
    base = np.clip(gray * 1.15, 0, 255).astype(np.uint8)
    base_rgb = np.repeat(base[..., None], 3, axis=-1)

    n_masks = int(masks.max())
    if n_masks == 0:
        return base_rgb

    # Deterministic pseudo-random hues in OpenCV HSV space
    idx = np.arange(n_masks + 1, dtype=np.uint16)
    hue = ((idx * 37) % 180).astype(np.uint8)
    hsv_lut = np.zeros((1, n_masks + 1, 3), dtype=np.uint8)
    hsv_lut[0, 1:, 0] = hue[1:]
    hsv_lut[0, 1:, 1] = 255
    hsv_lut[0, 1:, 2] = 255
    rgb_lut = cv2.cvtColor(hsv_lut, cv2.COLOR_HSV2RGB)[0]

    labels = np.clip(masks, 0, n_masks)
    color_map = rgb_lut[labels]

    out = base_rgb.copy()
    fg = labels > 0
    out[fg] = (0.35 * base_rgb[fg] + 0.65 * color_map[fg]).astype(np.uint8)
    return out


def draw_outlines_fast(img_rgb: np.ndarray, masks: np.ndarray) -> np.ndarray:
    """Draw red outlines on original image quickly."""
    outlines = _compute_outlines_fast(masks)
    out = img_rgb.copy()
    yy, xx = np.nonzero(outlines)
    out[yy, xx] = [255, 0, 0]
    return out


def _maybe_downscale(img_rgb: np.ndarray, max_side: int | None) -> np.ndarray:
    if max_side is None:
        return img_rgb
    h, w = img_rgb.shape[:2]
    m = max(h, w)
    if m <= max_side:
        return img_rgb
    scale = max_side / float(m)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)


def save_segmentation_figure_fast(img: np.ndarray, masks: np.ndarray, save_path: Path,name:str,ncells:int) -> None:
    """Save a 3-panel strip (original, outlines, masks) without matplotlib."""
    img_rgb = _to_uint8_rgb(img)
    outline_img = draw_outlines_fast(img_rgb, masks)
    overlay = mask_overlay_fast(img_rgb, masks)

    img_rgb = _maybe_downscale(img_rgb, VIZ_MAX_SIDE)
    outline_img = _maybe_downscale(outline_img, VIZ_MAX_SIDE)
    overlay = _maybe_downscale(overlay, VIZ_MAX_SIDE)

    panel = np.concatenate([img_rgb, outline_img, overlay], axis=1)

    save_path.parent.mkdir(parents=True, exist_ok=True)
    panel_bgr = cv2.cvtColor(panel, cv2.COLOR_RGB2BGR)
    cv2.imwrite(
        str(save_path/f"{name}_{ncells}.png"),
        panel_bgr,
        [int(cv2.IMWRITE_PNG_COMPRESSION), int(PNG_COMPRESSION)],
    )


In [82]:
# Accepted extensions
exts = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
image_paths = sorted([p for p in INPUT_DIR.rglob('*') if p.suffix.lower() in exts])
print('num_images:', len(image_paths))


num_images: 2208


In [83]:
results = []
failed_rows = []

for img_path in tqdm(image_paths, desc='StarDist inference'):
    try:
        img = iio.imread(img_path)
        labels, details = predict_one_image(img)

        rel = img_path.relative_to(INPUT_DIR)

        out_path = OUTPUT_DIR / rel.with_suffix('.stardist_labels.tif')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        imwrite(out_path, labels.astype(np.uint16), compression='zlib')

        viz_path = ''
        if SAVE_VIZ:
            viz_out = VIZ_DIR 
            save_segmentation_figure_fast(img, labels, viz_out,str(rel.stem),labels.max())
            viz_path = str(viz_out)

        results.append({
            'image': str(img_path),
            'output': str(out_path),
            'viz_output': viz_path,
            'shape_in': tuple(img.shape),
            'shape_out': tuple(labels.shape),
            'n_instances': int(labels.max()),
            'status': 'ok',
        })
    except Exception as e:
        err_msg = str(e)
        failed_rows.append({
            'image': str(img_path),
            'error': err_msg,
        })
        results.append({
            'image': str(img_path),
            'output': '',
            'viz_output': '',
            'shape_in': None,
            'shape_out': None,
            'n_instances': -1,
            'status': f'error: {err_msg}',
        })

len(results)


StarDist inference:   0%|          | 0/2208 [00:00<?, ?it/s]

2208

In [84]:
import pandas as pd

results_df = pd.DataFrame(results)
results_csv = OUTPUT_DIR / 'stardist_inference_summary.csv'
results_df.to_csv(results_csv, index=False)

failed_df = pd.DataFrame(failed_rows, columns=['image', 'error'])
failed_df.to_csv(FAILED_CSV, index=False)

display(results_df.head())
print('summary:', results_csv)
print('failed:', FAILED_CSV, '| n_failed =', len(failed_df))


,image,output,viz_output,shape_in,shape_out,n_instances,status
0,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,"(256, 256, 3)","(256, 256)",0,ok
1,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,"(256, 256, 3)","(256, 256)",2,ok
2,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,"(256, 256, 3)","(256, 256)",0,ok
3,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,"(256, 256, 3)","(256, 256)",0,ok
4,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,"(256, 256, 3)","(256, 256)",0,ok


summary: /share/lab_teng/trainee/tusharsingh/cell-seg/inference/stardist/outputs/RCC_5/stardist_inference_summary.csv
failed: /share/lab_teng/trainee/tusharsingh/cell-seg/inference/stardist/outputs/RCC_5/failed.csv | n_failed = 0


## Notes

- If GPU list is empty, this kernel is probably attached to a CPU/login node (not your GPU job node).
- For very large images, increase `N_TILES` (for example `(2,2)` or `(4,4)`) to reduce peak memory.
- Use `MODEL_NAME='2D_versatile_he'` for H&E brightfield; use `2D_versatile_fluo` for fluorescence nuclei.
- Visualization saving is optimized: OpenCV panel writer + JPEG + optional downscale (`VIZ_MAX_SIDE`).
- Per-image failures are logged to `failed.csv` and processing continues for remaining images.
